# PPE five-class training on Colab (S8)

**Read first.** This notebook runs the guarded harness (`scripts/train_ppe5.py`). Rules that still apply:
- Approval is for **this PoC only** (`config/training_authorization.yaml`). Nothing here approves deployment.
- The harness **refuses a real run if the network is reachable**, unless the owner records a waiver (cell 6). Colab cannot guarantee isolation by itself.
- The COCO weights are loaded only through `scripts/restricted_load.py` (static pickle scan plus allow-list).
- `models/best.pt` (Hafizqaim) is **not** uploaded or used.
- Accuracy from the CSS valid/test images is **sanity only**. A real result needs the independent evaluation set.

Before starting: upload `ppe5_colab_bundle.zip` (made by `scripts/make_colab_bundle.py`, about 150 MB) to your Google Drive, and set a **GPU runtime** (Runtime > Change runtime type).

## 1. GPU check

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import sys, platform; print(sys.version.split()[0], platform.platform())

## 2. Unpack the bundle and verify every file hash

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
BUNDLE = '/content/drive/MyDrive/ppe5_colab_bundle.zip'   # change if you uploaded it elsewhere
import zipfile, json, hashlib, os, pathlib
assert os.path.exists(BUNDLE), 'upload ppe5_colab_bundle.zip to Drive first'
!rm -rf /content/ws && mkdir -p /content && cd /content && unzip -q -o "$BUNDLE"
manifest = json.load(open('/content/bundle_manifest.json'))['files']
bad = [k for k, h in manifest.items() if hashlib.sha256(open('/content/'+k, 'rb').read()).hexdigest() != h]
assert not bad, f'hash mismatch: {bad[:5]}'
assert not os.path.exists('/content/ws/models/best.pt'), 'best.pt must not be on this host'
print(len(manifest), 'files verified; best.pt absent as intended')

## 3. Install pinned versions (matches the tested environment)

In [ ]:
PIN_TORCH = True   # parity with the tested environment; set False to keep Colab's own torch
!pip -q install ultralytics==8.3.70 pyyaml pytest
if PIN_TORCH:
    !pip -q install torch==2.6.0 torchvision==0.21.0
import torch; print('torch', torch.__version__, '| cuda', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')

## 4. Tests and dry run (must show no blockers)

In [ ]:
%cd /content/ws/Renewi_EHS_Lighthouse
!python -m pytest -q backend/tests/test_train_ppe5.py backend/tests/test_restricted_load.py backend/tests/test_evaluate_ppe5.py 2>&1 | tail -5
!python scripts/train_ppe5.py --dry-run --config config/train_ppe5.colab.yaml

## 5. Network isolation
The harness will refuse a real run while the network is reachable. Two ways forward:
1. **Preferred:** run the training inside a network namespace (`unshare -rn`). The cell below tests whether Colab allows it.
2. **Waiver:** the project owner may waive isolation for this host. Set `WAIVE = True` **and** fill in the name and date. Then also record the waiver in the repo's `config/training_authorization.yaml`. The restricted loader stays in force either way.

In [ ]:
import subprocess
r = subprocess.run('unshare -rn python3 -c "import socket;socket.create_connection((\'1.1.1.1\',53),timeout=3)"', shell=True, capture_output=True)
NS_OK = (r.returncode != 0 and b'Operation not permitted' not in r.stderr and b'unshare' not in r.stderr)
print('network namespace usable and blocks the network:', NS_OK)
PREFIX = 'unshare -rn ' if NS_OK else ''

WAIVE = False                 # the owner sets this to True deliberately
WAIVED_BY, WAIVED_ON = '', '' # required if WAIVE is True
if not NS_OK and WAIVE:
    assert WAIVED_BY and WAIVED_ON, 'fill in name and date'
    p = '/content/ws/Renewi_EHS_Lighthouse/config/training_authorization.yaml'
    t = open(p).read().replace('network_isolation_waived: false', f'network_isolation_waived: true\nnetwork_isolation_waived_by: "{WAIVED_BY}"\nnetwork_isolation_waived_on: "{WAIVED_ON}"')
    open(p, 'w').write(t); print('waiver recorded in the Colab copy only')
elif not NS_OK:
    print('No isolation and no waiver: the harness will refuse a real run. That is the intended safe default.')

## 6. Smoke test (1 epoch, tiny subset). Must pass before the full run

In [ ]:
!{PREFIX} python scripts/train_ppe5.py --smoke --config config/train_ppe5.colab.yaml --run-name colab-smoke

## 7. Copy results to Drive continuously (protects against Colab disconnects)

In [ ]:
!mkdir -p /content/drive/MyDrive/ppe5_runs
import subprocess
SYNC = subprocess.Popen('while true; do rsync -a /content/ws/runs/ /content/drive/MyDrive/ppe5_runs/ ; sleep 120; done', shell=True)
print('background sync started, pid', SYNC.pid)

## 8. Full run (about 100 epochs). **Only the owner starts this.**
Stage 1: 10 epochs with the backbone frozen. Stage 2: 90 epochs, everything unfrozen. Fixed schedule, final-epoch weights kept. If Colab disconnects mid-run the run cannot be resumed by this harness: copy what exists from Drive and start a new run name.

In [ ]:
RUN_NAME = 'colab-full-1'
!{PREFIX} python scripts/train_ppe5.py --full --config config/train_ppe5.colab.yaml --run-name {RUN_NAME}

## 9. Record the results

In [ ]:
import json, glob
r = json.load(open(f'/content/ws/runs/{RUN_NAME}/run_record.json'))
print('final checkpoint:', r['final_checkpoint'])
print('final checkpoint SHA-256:', r['final_checkpoint_sha256'])
print('protected checkpoint before/after:', r['protected_sha256_before'], r['protected_sha256_after'])
print('network:', r['evidence'].get('network'))
!rsync -a /content/ws/runs/{RUN_NAME}/ /content/drive/MyDrive/ppe5_runs/{RUN_NAME}/
print('Copied to Drive: ppe5_runs/' + RUN_NAME)
# Next: add the new checkpoint's own class map to config/classes.yaml (ids 0-4) and evaluate on the independent set.
# Do NOT report CSS valid/test numbers as the result.